## About

Explore the functionality of various tools that are available in the medea.
This helps with understanding how to use them for my work.

In [2]:
import os
import json
import torch
import pandas as pd


In [2]:

base_url = "https://api.platform.opentargets.org/api/v4/graphql"

open_target_query_string = """
    query target($diseaseName: String!){
        search(queryString: $diseaseName){
        hits{
            id
            entity
            category
            name
            description
        }
    }
}
"""


In [3]:
def load_pinnacle_ppi(
    cell_type: str,
    embed_path: str = None, 
    weights_only=False
):
    """
    Load the PINNACLE PPI embeddings for a specific cell type from a specified path.
    
    Uses the same normalization and matching logic as celltype_avaliability_checker() to ensure consistency.
    Accepts multiple cell type formats and normalizes them automatically.
    
    Args:
        cell_type (str): The specific cell type to load embeddings for. Accepts multiple formats:
                        - Standardized: 'cd4_positive_alpha_beta_memory_t_cell'
                        - Raw: 'cd4-positive,_alpha-beta_memory_t_cell'
                        - User-friendly: 'CD4+ memory T cell'
                        All formats are normalized to the same internal representation.
        embed_path (str): Path to the PPI embeddings file. If None, uses MEDEADB_PATH environment variable.
        weights_only (bool): Whether to load weights only or the full state.
    
    Returns:
        dict: A dictionary of PPI embeddings for the specified cell type with gene name as key 
            and cell type-specific gene embedding as value (torch.Tensor). Returns empty dict if cell type not found.
    """
    # Set default embed_path if not provided
    if embed_path is None:
        embed_path = os.path.join(_get_medeadb_path(), 'pinnacle_embeds/ppi_embed_dict.pth')
    
    # Load the full PPI dictionary from the specified path
    ppi_dict = torch.load(embed_path, weights_only=weights_only)
    
    # Normalize cell type using same function as checker
    def _normalize(s):
        return s.replace(",", "").replace("-", "_").replace(" ", "_").replace("+", "_positive").replace("α", "alpha").replace("β", "beta").lower()
    
    def _format_display(s):
        """Format for clean, consistent display."""
        formatted = s.replace(",", "").replace("-", "_").replace(" ", "_")
        while "__" in formatted:
            formatted = formatted.replace("__", "_")
        return formatted.lower()
    
    formalized_cell_type = _normalize(cell_type)
    
    # First pass: exact match
    for cell_key in ppi_dict.keys():
        formalized_key = _normalize(cell_key)
        if formalized_key == formalized_cell_type:
            formatted_name = _format_display(cell_key)
            print(f"[load_pinnacle_ppi] ✓ Loaded {len(ppi_dict[cell_key])} genes for cell type: '{formatted_name}'", flush=True)
            return ppi_dict[cell_key]
        
    # Second pass: fuzzy matching with same logic as checker
    from thefuzz import fuzz
    best_match = None
    best_score = 0
    
    for cell_key in ppi_dict.keys():
        candidate_norm = _normalize(cell_key)
        
        # Compute similarity score using multiple strategies
        token_sort = fuzz.token_sort_ratio(formalized_cell_type, candidate_norm)
        token_set = fuzz.token_set_ratio(formalized_cell_type, candidate_norm)
        
        # Combined score
        score = token_sort * 0.7 + token_set * 0.3
        
        # Token overlap boost
        query_tokens = set(formalized_cell_type.split('_'))
        candidate_tokens = set(candidate_norm.split('_'))
        if query_tokens and candidate_tokens:
            intersection = len(query_tokens.intersection(candidate_tokens))
            union = len(query_tokens.union(candidate_tokens))
            jaccard = intersection / union if union > 0 else 0
            score += jaccard * 15
        
        if score > best_score:
            best_score = score
            best_match = cell_key
    
    # Return best match if score is reasonable
    if best_match and best_score >= 60:
        formatted_name = _format_display(best_match)
        print(f"[load_pinnacle_ppi] ⚠ Cell type '{cell_type}' matched to '{formatted_name}' (score: {best_score:.0f})", flush=True)
        print(f"[load_pinnacle_ppi] ✓ Loaded {len(ppi_dict[best_match])} genes for matched cell type", flush=True)
        return ppi_dict[best_match]
    
    # If no good match found, return empty dict
    print(f"[load_pinnacle_ppi] ✗ ERROR: Cell type '{cell_type}' not found in PINNACLE embeddings (best match score: {best_score:.0f}).", flush=True)
    print(f"[load_pinnacle_ppi] → Please use the celltype_avaliability_checker to find valid cell types.", flush=True)
    return {}


In [4]:
## ppi_embed_dict returns gene as key and 128 dim embedding as vector

ppi_embed_dict = load_pinnacle_ppi(cell_type = "cd4_positive_alpha_beta_memory_t_cell", 
                                    embed_path = os.path.join("..","database","MedeaDB","pinnacle_embeds","ppi_embed_dict.pth"), 
                                    weights_only = False)

[load_pinnacle_ppi] ✓ Loaded 1067 genes for cell type: 'cd4_positive_alpha_beta_memory_t_cell'


In [5]:
list(ppi_embed_dict.items())[10][-1].shape

torch.Size([128])

## Check the open target API access

In [6]:

import requests
import re
def search_disease_open_target(disease_name):
    # Set variables object of arguments to be passed to endpoint
    relavent_entities = None
    variables = {"diseaseName": disease_name}
    for i in range(5):
        try:
            # Perform POST request and check status code of response
            r = requests.post(base_url, json={"query": open_target_query_string, "variables": variables})
            # Transform API response from JSON into Python dictionary and print in console
            api_response = json.loads(r.text)
            relavent_entities = api_response['data']['search']['hits']
        except Exception as e:
            print(f"[OpenTarget] Bad OpenTarget API response, retrying...")
            continue
        if relavent_entities is not None: break

    if relavent_entities is None:
        raise ValueError(f"[OpenTarget] No relevant entities found for {disease_name}")
    # Check the top 5 entities
    filtered_entities = [ e for e in relavent_entities if e['entity'] == 'disease']
    return filtered_entities

In [7]:
search_disease_open_target(disease_name="rheumatoid arthritis")

[{'id': 'EFO_0000685',
  'entity': 'disease',
  'category': ['musculoskeletal or connective tissue disease',
   'immune system disease'],
  'name': 'rheumatoid arthritis',
  'description': 'A chronic, systemic autoimmune disorder characterized by inflammation in the synovial membranes and articular surfaces. It manifests primarily as a symmetric, erosive polyarthritis that spares the axial skeleton and is typically associated with the presence in the serum of rheumatoid factor.'},
 {'id': 'HP_0001370',
  'entity': 'disease',
  'category': ['phenotype'],
  'name': 'Rheumatoid arthritis',
  'description': 'Inflammatory changes in the synovial membranes and articular structures with widespread fibrinoid degeneration of the collagen fibers in mesenchymal tissues, as well as atrophy and rarefaction of bony structures.'},
 {'id': 'EFO_0009460',
  'entity': 'disease',
  'category': ['musculoskeletal or connective tissue disease',
   'immune system disease'],
  'name': 'ACPA-negative rheumatoi

## Study the target protein retrival

In [28]:
import os
os.chdir("..")
from medea.tool_space.action_functions import *

In [31]:
os.chdir("Medea")

In [7]:
target_set = load_disease_targets(disease_name="rheumatoid arthritis")

[load_disease_targets] Querying OpenTargets API for disease: rheumatoid arthritis (EFO_0000685)
[load_disease_targets] Found 4967 total target associations from API
[load_disease_targets] Fetching page 2 (retrieved 1872 targets so far)...
[load_disease_targets] Retrieved 1872 targets from OpenTargets API after filtering


In [8]:
target_set


{'IFNG',
 'LEMD2',
 'ADRB2',
 'PCDH20',
 'CHD9',
 'SPEF2',
 'AHNAK2',
 'STAT3',
 'SYF2',
 'TOR3A',
 'TMEM235',
 'FUT2',
 'RGS6',
 'CX3CL1',
 'RNASET2',
 'TLE3',
 'LAMTOR4',
 'METRN',
 'DEXI',
 'DDR1',
 'SAG',
 'TUBD1',
 'TSNARE1',
 'CCNG2',
 'NFKBIA',
 'LRRC40',
 'GRPEL2',
 'TAX1BP1',
 'TNIP2',
 'BARX1',
 'SOCS1',
 'HECTD4',
 'PLB1',
 'THADA',
 'STARD3',
 'KPNB1',
 'LRRC32',
 'PSTPIP1',
 'PHB2',
 'EIF3CL',
 'B3GAT1',
 'NPEPPS',
 'ETV1',
 'PIGH',
 'HMGA1',
 'HUS1B',
 'CDK2',
 'TLR4',
 'LCE3C',
 'DNAJC5B',
 'APOBR',
 'TPP2',
 'SMTNL2',
 'NDUFV1',
 'MERTK',
 'SPNS1',
 'TENM4',
 'FRMD4B',
 'LHFPL6',
 'DELE1',
 'C1QTNF2',
 'PRR15L',
 'ZMAT3',
 'ZC3H11A',
 'HDHD2',
 'PLCL2',
 'RAB38',
 'TPH2',
 'CSF2RB',
 'FGFR2',
 'RMDN2',
 'CTLA4',
 'SF3A3',
 'EGR2',
 'ANKMY2',
 'RPL15',
 'IKZF1',
 'ZNF534',
 'SLC35G5',
 'SULT1A3',
 'MYCBP2',
 'SELE',
 'PDE3B',
 'PDE8B',
 'RSPH3',
 'TCF7',
 'ATRX',
 'ENTR1',
 'CD83',
 'PLEKHG7',
 'XCR1',
 'CD28',
 'BTN2A1',
 'CD52',
 'DBP',
 'IMPDH1',
 'MEAK7',
 'FGFR3',
 

In [4]:
os.getcwd()

'c:\\Harinee_D\\IIT_PostBacc\\AgenticAI\\Medea'

In [16]:
## Cross check the target set from OT with the evaluation datasets
cwd = os.path.join(os.getcwd(),"AgenticAI", "MEDEA")
ra_42_df = pd.read_csv(os.path.join(cwd, "evaluation", "targetID", "source", "targetid-ra-42.csv"))
ra_42_df.head()
celltype_to_y = ra_42_df.groupby('celltype')['y'].apply(list).to_dict()
celltype_to_y

{'classical monocyte': ['IL1B',
  'ADAM17',
  'AHR',
  'HLA-E',
  'HLA-DRA',
  'HIF1A',
  'TLE3',
  'RPLP1',
  'MTPAP',
  'IFNGR2',
  'ZEB2',
  'NFKBIA',
  'CDC42EP3',
  'ZEB2',
  'MTPAP',
  'CDC42EP3',
  'RPLP1',
  'IL1B',
  'NFKBIA',
  'TLE3'],
 'effector memory cd4 positive alpha beta t cell': ['CORO1A',
  'CD2',
  'HLA-C',
  'HLA-A',
  'RPL3',
  'CD52',
  'LTB',
  'CD3E',
  'JAK3',
  'RPLP1',
  'MT-ND3',
  'GPSM3',
  'S100A10',
  'RPL21',
  'PPIA',
  'IL6ST',
  'COX4I1',
  'PRDM1',
  'DDX6',
  'TNFAIP3'],
 'myeloid dendritic cell': ['RPS26',
  'COX4I1',
  'CD52',
  'HLA-E',
  'RPL21',
  'RPL3',
  'RPLP1',
  'LTB',
  'ACTB',
  'RPS26',
  'ACTB',
  'COX4I1',
  'RPL3',
  'CD52',
  'HLA-E',
  'RPL21',
  'LTB',
  'RPLP1',
  'RPLP1',
  'RPS26'],
 'naive b cell': ['MT-ND3',
  'MS4A1',
  'FOS',
  'NFKBIA',
  'IRF1',
  'YPEL5',
  'RPS26',
  'NFKBIA',
  'IRF1',
  'YPEL5',
  'MS4A1',
  'FOS',
  'RPS26',
  'MT-ND3',
  'IRF1',
  'MS4A1',
  'RPS26',
  'YPEL5',
  'MT-ND3',
  'NFKBIA'],
 'naive th

In [6]:
for celltype, targets in celltype_to_y.items():
    print(f"Cell type: {celltype}")
    overlap = set(targets) & target_set
    percentage_overlap = (len(overlap) / len(targets)) * 100 if targets else 0
    print(f"Percentage of overlap with target_set: {percentage_overlap:.2f}%")

Cell type: classical monocyte
Percentage of overlap with target_set: 40.00%
Cell type: effector memory cd4 positive alpha beta t cell
Percentage of overlap with target_set: 70.00%
Cell type: myeloid dendritic cell
Percentage of overlap with target_set: 25.00%
Cell type: naive b cell
Percentage of overlap with target_set: 30.00%
Cell type: naive thymus derived cd4 positive alpha beta t cell
Percentage of overlap with target_set: 55.00%
Cell type: naive thymus derived cd8 positive alpha beta t cell
Percentage of overlap with target_set: 50.00%
Cell type: natural killer cell
Percentage of overlap with target_set: 15.00%


## Next: For RA, what are the celltypes that can be targetted. is there any tool that talks about this?

User has to provide the celltype via the input query

In [1]:
from tool_space.open_alex import *

[Warning] ToolUniverse not available: No module named 'tooluniverse'
[Warning] Install tooluniverse to use Medea tools: pip install tooluniverse


In [12]:
import sys
import dotenv
dotenv.load_dotenv('.env')
#sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
#from tool_space.open_alex import search_openalex_papers, paper_search_from_openalex

# Test new semantic scholar compatible function
print("=== Testing search_openalex_papers (semantic scholar format) ===")
papers, keywords = search_openalex_papers("MCF7 Breast Cancer", max_results=3, open_access=True)
print(f"Keywords used: {keywords}")
print(f"Number of papers: {len(papers)}")
if papers:
    print(f"First paper format:")
    for key, value in papers[0].items():
        print(f"  {key}: {value}")

print("\n=== Testing paper_search_from_openalex (legacy format) ===")
result = paper_search_from_openalex("MCF7 Breast Cancer", max_results=2)
if isinstance(result, list) and result:
    print(f"Number of papers: {len(result)}")
    print(f"First paper keys: {list(result[0].keys())}")
else:
    print(f"Result: {result}")

=== Testing search_openalex_papers (semantic scholar format) ===
[INFO] Loading FlagEmbedding library (may download reranker models)...
[SEARCH_OPENALEX] WARNING: Keyword generation failed (No module named 'FlagEmbedding'), using question directly
[SEARCH_OPENALEX] DEBUG: Processing keyword 1/1: 'MCF7 Breast Cancer'
[OpenAlex] Retrieved 3 papers for keywords: 'MCF7 Breast Cancer'
[SEARCH_OPENALEX] SUCCESS: Found 3 papers for keyword: 'MCF7 Breast Cancer'
[SEARCH_OPENALEX] SUMMARY: 1/1 queries successful, 3 unique papers found
Keywords used: ['MCF7 Breast Cancer']
Number of papers: 3
First paper format:
  semantic_scholar_id: https://openalex.org/W2276797747
  type: openalex_abstract
  year: 2016
  authors: [{'authorId': 'https://openalex.org/A5022128827', 'name': 'Jhih-Pu Syu'}, {'authorId': 'https://openalex.org/A5066099267', 'name': 'Jen‐Tsan Chi'}, {'authorId': 'https://openalex.org/A5022430655', 'name': 'Hsiu‐Ni Kung'}]
  title: Nrf2 is the key to chemotherapy resistance in MCF7 br

In [14]:
keywords

['MCF7 Breast Cancer']

In [13]:
papers

[{'semantic_scholar_id': 'https://openalex.org/W2276797747',
  'type': 'openalex_abstract',
  'year': 2016,
  'authors': [{'authorId': 'https://openalex.org/A5022128827',
    'name': 'Jhih-Pu Syu'},
   {'authorId': 'https://openalex.org/A5066099267', 'name': 'Jen‐Tsan Chi'},
   {'authorId': 'https://openalex.org/A5022430655', 'name': 'Hsiu‐Ni Kung'}],
  'title': 'Nrf2 is the key to chemotherapy resistance in MCF7 breast cancer cells under hypoxia',
  'text': 'Hypoxia leads to reactive oxygen species (ROS) imbalance, which is proposed to associate with drug resistance and oncogenesis. Inhibition of enzymes of antioxidant balancing system in tumor cells was shown to reduce chemoresistance under hypoxia. However, the underlying mechanism remains unknown. The key regulator of antioxidant balancing system is nuclear factor erythroid 2-related factor 2 (NFE2L2, Nrf2). In this study, we showed that hypoxia induced ROS production and increased the Nrf2 activity. Nrf2 activation increased level

In [24]:
standardize_disease_name("Rheumatoid Arthritis")

'Rheumatoid Arthritis'

In [28]:
standardize_disease_name("Alzheimer's disease")

'Alzheimer Disease'

In [27]:
get_efo_id("Rheumatoid Arthritis")

'EFO:0000685'

In [29]:
get_efo_id("Alzheimer's disease")

'MONDO:0004975'

In [31]:
search_disease_efo("Alzheimer's disease")

'MONDO_0004975'

In [33]:
load_disease_targets(disease_name="Rheumatoid Arthritis")

[load_disease_targets] Querying OpenTargets API for disease: Rheumatoid Arthritis (EFO_0000685)
[load_disease_targets] Found 4967 total target associations from API
[load_disease_targets] Fetching page 2 (retrieved 1872 targets so far)...
[load_disease_targets] Retrieved 1872 targets from OpenTargets API after filtering


{'IFNG',
 'LEMD2',
 'ADRB2',
 'PCDH20',
 'CHD9',
 'SPEF2',
 'AHNAK2',
 'STAT3',
 'SYF2',
 'TOR3A',
 'TMEM235',
 'FUT2',
 'RGS6',
 'CX3CL1',
 'RNASET2',
 'TLE3',
 'LAMTOR4',
 'METRN',
 'DEXI',
 'DDR1',
 'SAG',
 'TUBD1',
 'TSNARE1',
 'CCNG2',
 'NFKBIA',
 'LRRC40',
 'GRPEL2',
 'TAX1BP1',
 'TNIP2',
 'BARX1',
 'SOCS1',
 'HECTD4',
 'PLB1',
 'THADA',
 'STARD3',
 'KPNB1',
 'LRRC32',
 'PSTPIP1',
 'PHB2',
 'EIF3CL',
 'B3GAT1',
 'NPEPPS',
 'ETV1',
 'PIGH',
 'HMGA1',
 'HUS1B',
 'CDK2',
 'TLR4',
 'LCE3C',
 'DNAJC5B',
 'APOBR',
 'TPP2',
 'SMTNL2',
 'NDUFV1',
 'MERTK',
 'SPNS1',
 'TENM4',
 'FRMD4B',
 'LHFPL6',
 'DELE1',
 'C1QTNF2',
 'PRR15L',
 'ZMAT3',
 'ZC3H11A',
 'HDHD2',
 'PLCL2',
 'RAB38',
 'TPH2',
 'CSF2RB',
 'FGFR2',
 'RMDN2',
 'CTLA4',
 'SF3A3',
 'EGR2',
 'ANKMY2',
 'RPL15',
 'IKZF1',
 'ZNF534',
 'SLC35G5',
 'SULT1A3',
 'MYCBP2',
 'SELE',
 'PDE3B',
 'PDE8B',
 'RSPH3',
 'TCF7',
 'ATRX',
 'ENTR1',
 'CD83',
 'PLEKHG7',
 'XCR1',
 'CD28',
 'BTN2A1',
 'CD52',
 'DBP',
 'IMPDH1',
 'MEAK7',
 'FGFR3',
 

In [34]:
load_disease_targets(disease_name="Alzheimer's disease",attributes=["otGeneticsPortal"])

[load_disease_targets] Querying OpenTargets API for disease: Alzheimer's disease (MONDO_0004975)
[load_disease_targets] Found 12995 total target associations from API
[load_disease_targets] Fetching page 2 (retrieved 1115 targets so far)...
[load_disease_targets] Fetching page 3 (retrieved 1683 targets so far)...
[load_disease_targets] Fetching page 4 (retrieved 1683 targets so far)...
[load_disease_targets] Fetching page 5 (retrieved 1683 targets so far)...
[load_disease_targets] Retrieved 1683 targets from OpenTargets API after filtering


{'SRRM3',
 'COL11A1',
 'ACKR2',
 'CBY1',
 'PILRB',
 'KCNQ5',
 'EPRS1',
 'ZNF146',
 'EEF1AKMT2',
 'KCNIP2',
 'SGPP1',
 'TRIM41',
 'ACER1',
 'POTEC',
 'TOR3A',
 'PRPF40B',
 'FAM161B',
 'SYNPO2L',
 'KCNJ18',
 'SSRP1',
 'SUPT4H1',
 'PRELID2',
 'OLA1',
 'SHISA4',
 'GLDC',
 'STRIP2',
 'C6orf118',
 'RABGEF1',
 'SUSD3',
 'CLTA',
 'DLG2',
 'TAX1BP1',
 'CACNB2',
 'HECTD4',
 'EIF4EBP2',
 'C12orf50',
 'KCNJ12',
 'KRTAP13-3',
 'ZFR2',
 'GLYATL3',
 'WDR70',
 'SLC12A2',
 'LHX5',
 'BIN1',
 'NLRP7',
 'SUPT3H',
 'HNRNPA0',
 'ETV1',
 'DEAF1',
 'HMGA1',
 'LRRC4C',
 'KRTAP19-3',
 'NRXN1',
 'PHKB',
 'APOBR',
 'CAPZA2',
 'SPNS1',
 'MERTK',
 'TMEM185B',
 'SUZ12',
 'NLGN1',
 'SYNE2',
 'SETDB1',
 'CCNYL1B',
 'ADARB1',
 'PRKDC',
 'KRTAP13-4',
 'TRPM7',
 'STYX',
 'SRBD1',
 'IGFBP3',
 'C12orf43',
 'EBLN2',
 'PTPRS',
 'LRRC37A',
 'TPH2',
 'ABHD17A',
 'RMDN2',
 'BANF1',
 'ARHGAP24',
 'IQCF2',
 'TMX4',
 'ANKMY2',
 'ADAM10',
 'TACC2',
 'PSG9',
 'MS4A6E',
 'SLC52A3',
 'OAS1',
 'NR1H3',
 'PDE3B',
 'ZNF526',
 'TCF7L2',
 

In [36]:
load_disease_targets(disease_name="Alzheimer's disease",attributes= "chembl")

[load_disease_targets] Querying OpenTargets API for disease: Alzheimer's disease (MONDO_0004975)
[load_disease_targets] Found 12995 total target associations from API
[load_disease_targets] Fetching page 2 (retrieved 2992 targets so far)...
[load_disease_targets] Fetching page 3 (retrieved 5985 targets so far)...
[load_disease_targets] Fetching page 4 (retrieved 8974 targets so far)...
[load_disease_targets] Fetching page 5 (retrieved 11949 targets so far)...
[load_disease_targets] Retrieved 12850 targets from OpenTargets API after filtering


{'IFNG',
 'LEMD2',
 'NCAM2',
 'SIGLEC5',
 'RHPN1',
 'MGME1',
 'VPS28',
 'UROC1',
 'ZNF146',
 'CRELD1',
 'SGPP1',
 'SAMD8',
 'PCDH20',
 'ATP7A',
 'AHNAK2',
 'TRIM41',
 'MIER1',
 'ZNF200',
 'MFSD2A',
 'TPRG1L',
 'SV2A',
 'LYSMD2',
 'PCYT1B',
 'SYNPO2L',
 'SLC26A4',
 'GSAP',
 'DCTN4',
 'ACBD6',
 'AMELX',
 'SUPT4H1',
 'SHC2',
 'CTHRC1',
 'BIRC8',
 'LPAL2',
 'CCDC51',
 'TLE3',
 'CDK4',
 'IGHA1',
 'TAAR5',
 'FRMD6',
 'NUCB1',
 'SUSD3',
 'TGFBR3',
 'CLTA',
 'DLG2',
 'TMPRSS15',
 'LMX1B',
 'IFNA17',
 'CPB1',
 'MYO1E',
 'C12orf50',
 'KCNJ12',
 'IGHV3-30',
 'MRPL42',
 'RLIG1',
 'PLB1',
 'MTMR10',
 'TCEAL1',
 'CCNG1',
 'THADA',
 'STARD3',
 'SNF8',
 'PLXNB2',
 'LTK',
 'BACE1',
 'BIN1',
 'VOPP1',
 'MMP2',
 'CCL18',
 'RPL5',
 'MYO9A',
 'ENSG00000279453',
 'NID1',
 'TARBP2',
 'DEAF1',
 'DHRS4',
 'ACOX3',
 'SCAP',
 'CDK2',
 'KRTAP19-3',
 'S100P',
 'NAA16',
 'PRTG',
 'PMP22',
 'LCE3C',
 'PHKB',
 'TPP2',
 'MINK1',
 'PHF20',
 'CR2',
 'BLVRB',
 'FSTL5',
 'SUZ12',
 'GOT2',
 'NOXRED1',
 'SYNE2',
 'NUTM1',
 

In [37]:
get_gene_synonyms("APP")

['AAA',
 'ABETA',
 'ABPP',
 'AD1',
 'APPI',
 'CTFgamma',
 'CVAP',
 'PN-II',
 'PN2',
 'alpha-sAPP',
 'preA4']

In [5]:
from medea.tool_space.depmap import *

In [43]:
MEDEADB_PATH = "C:\Harinee_D\IIT_PostBacc\AgenticAI\Medea\database\MedeaDB"

In [6]:
compute_depmap24q2_gene_correlations("APP","PSEN1")

[DEPMAP] ANALYSIS: Initiating cell viability correlation analysis: APP ↔ PSEN1
[DEPMAP] ANALYSIS: Gene symbols standardized: APP, PSEN1
[DEPMAP] ANALYSIS: Loading DepMap 24Q2 correlation matrix from: C:/Harinee_D/IIT_PostBacc/AgenticAI/Medea/database/MedeaDB\depmap_24q2
[DEPMAP] ANALYSIS: Successfully loaded correlation data for 18,443 genes
[DEPMAP] ANALYSIS: Gene pair validation successful - both genes found in dataset
[DEPMAP] ANALYSIS: Computing CERES-based correlation across 1,320 cancer cell lines
[DEPMAP] ANALYSIS: Correlation coefficient: 0.0159
[DEPMAP] ANALYSIS: Statistical significance: p = 5.65e-01
[DEPMAP] ANALYSIS: Multiple testing correction: adj_p = 7.34e-01
[DEPMAP] ANALYSIS: Biological interpretation: No reliable evidence for correlated cell viability effects between APP and PSEN1
[DEPMAP] ANALYSIS: Analysis completed successfully - correlation: 0.0159, significance: not statistically significant


(0.015857279300689697, 0.564876139163971, 0.7344430685043335)

In [7]:
compute_depmap24q2_gene_correlations("APP","SEPT8")

[DEPMAP] ANALYSIS: Initiating cell viability correlation analysis: APP ↔ SEPT8
[DEPMAP] ANALYSIS: Gene symbols standardized: APP, SEPT8
[DEPMAP] ANALYSIS: Loading DepMap 24Q2 correlation matrix from: C:/Harinee_D/IIT_PostBacc/AgenticAI/Medea/database/MedeaDB\depmap_24q2
[DEPMAP] ANALYSIS: Successfully loaded correlation data for 18,443 genes
[DEPMAP] ERROR: Genes not found in DepMap dataset: SEPT8
[DEPMAP] INFO: Available genes: 18,443 total in DepMap 24Q2
[DEPMAP] ERROR: Gene availability error: 'Gene(s) not available in DepMap correlation matrix: SEPT8'
[DEPMAP] INFO: Recommendation: Verify gene symbols or check DepMap gene coverage


(None, None, None)

#### wikipathways

In [10]:
from medea.tool_space.enrichr import *
analyze_pathway_interaction("APP", "PSEN1")

[PATHWAY] INSIGHT: Analyzing pathway interactions for APP + PSEN1
[PATHWAY] INSIGHT: Strong evidence found: 'Inclusion Body Myositis WP5120' (p=2.75e-07)
[PATHWAY] INSIGHT: Interaction detected: 5 shared terms, score=24.22
[PATHWAY] INSIGHT: Biological mechanisms identified: general pathway interaction
[PATHWAY] INSIGHT: Confidence 'high' based on: multiple shared terms (5), high statistical significance
[PATHWAY] INSIGHT: Pathway analysis complete: high confidence, 1 mechanism types


('Genes interact through 5 shared pathway(s) with high confidence',
 'high',
 ['general_pathway'],
 ['Inclusion Body Myositis WP5120',
  'Neurogenesis Regulation In The Olfactory Epithelium WP5265',
  'DYRK1A Involvement Regarding Cell Prolif In Brain Development WP5180',
  'Alzheimer 39 S Disease And miRNA Effects WP2059',
  'Alzheimer 39 S Disease WP5124'])

#### others: reactome, MSigDB, GO molecular functions, GO biological process

In [12]:
analyze_reactome_interaction("APP", "PSEN1")

[REACTOME] INSIGHT: Analyzing biochemical interactions for APP + PSEN1
[REACTOME] INSIGHT: Strong evidence found: 'Extracellular Matrix Organization' (p=2.24e-04)
[REACTOME] INSIGHT: Interaction detected: 1 shared terms, score=3.65
[REACTOME] INSIGHT: Molecular mechanisms identified: general biochemical interaction
[REACTOME] INSIGHT: Confidence 'very_low' based on: single shared term
[REACTOME] INSIGHT: Reactome analysis complete: very_low confidence, molecular interaction detected


('Genes interact through 1 shared biochemical reaction(s) with very_low confidence',
 'very_low',
 ['biochemical_interaction'],
 ['Extracellular Matrix Organization'])

In [13]:
analyze_hallmark_interaction("APP", "PSEN1")

[HALLMARK] INSIGHT: Analyzing cancer hallmark processes for APP + PSEN1
[HALLMARK] INSIGHT: Strong evidence found: 'Apoptosis' (p=6.44e-05)
[HALLMARK] INSIGHT: Interaction detected: 1 shared terms, score=4.19
[HALLMARK] INSIGHT: Cancer biology roles identified: cell death regulation
[HALLMARK] INSIGHT: Confidence 'low' based on: single shared term
[HALLMARK] INSIGHT: Hallmark analysis complete: low confidence, cancer relevance established


('Genes co-regulate 1 cancer hallmark process(es) with low confidence',
 'low',
 ['apoptotic_regulation'],
 ['Apoptosis'])

In [14]:
analyze_function_interaction("APP", "PSEN1")

[GO_FUNCTION] INSIGHT: Analyzing molecular functions for APP + PSEN1
[GO_FUNCTION] INSIGHT: No shared terms detected between gene pair
[GO_FUNCTION] INSIGHT: Genes have distinct molecular function profiles


('Genes do not share molecular functions', 'none', [], [])

In [15]:
analyze_process_interaction("APP", "PSEN1")

[GO_PROCESS] INSIGHT: Analyzing biological processes for APP + PSEN1
[GO_PROCESS] INSIGHT: Strong evidence found: 'Negative Regulation Of Low-Density Lipoprotein Receptor Activity (GO:1905598)' (p=7.50e-08)
[GO_PROCESS] INSIGHT: Interaction detected: 5 shared terms, score=32.81
[GO_PROCESS] INSIGHT: Biological processes identified: developmental biology
[GO_PROCESS] INSIGHT: Confidence 'high' based on: multiple shared terms (5), high statistical significance
[GO_PROCESS] INSIGHT: Process analysis complete: high confidence, process co-participation detected


('Genes participate in 5 shared biological process(es) with high confidence',
 'high',
 ['developmental_process'],
 ['Negative Regulation Of Low-Density Lipoprotein Receptor Activity (GO:1905598)',
  'Neuron Projection Maintenance (GO:1990535)',
  'Astrocyte Activation (GO:0048143)',
  'Regulation Of Amyloid Fibril Formation (GO:1905906)',
  'Astrocyte Development (GO:0014002)'])

#### all together

In [16]:
analyze_comprehensive_interaction("APP", "PSEN1")


=== COMPREHENSIVE INTERACTION ANALYSIS ===
Analyzing gene pair: (APP, PSEN1)

--- PATHWAY ANALYSIS ---
[PATHWAY] INSIGHT: Analyzing pathway interactions for APP + PSEN1
[PATHWAY] INSIGHT: Strong evidence found: 'Inclusion Body Myositis WP5120' (p=2.75e-07)
[PATHWAY] INSIGHT: Interaction detected: 5 shared terms, score=24.22
[PATHWAY] INSIGHT: Biological mechanisms identified: general pathway interaction
[PATHWAY] INSIGHT: Confidence 'high' based on: multiple shared terms (5), high statistical significance
[PATHWAY] INSIGHT: Pathway analysis complete: high confidence, 1 mechanism types
Summary: Genes interact through 5 shared pathway(s) with high confidence
Confidence: high
Mechanisms: ['general_pathway']
Key Evidence: ['Inclusion Body Myositis WP5120', 'Neurogenesis Regulation In The Olfactory Epithelium WP5265', 'DYRK1A Involvement Regarding Cell Prolif In Brain Development WP5180']

--- REACTOME ANALYSIS ---
[REACTOME] INSIGHT: Analyzing biochemical interactions for APP + PSEN1
[REA

{'pathway': ('Genes interact through 5 shared pathway(s) with high confidence',
  'high',
  ['general_pathway'],
  ['Inclusion Body Myositis WP5120',
   'Neurogenesis Regulation In The Olfactory Epithelium WP5265',
   'DYRK1A Involvement Regarding Cell Prolif In Brain Development WP5180',
   'Alzheimer 39 S Disease And miRNA Effects WP2059',
   'Alzheimer 39 S Disease WP5124']),
 'reactome': ('Genes interact through 1 shared biochemical reaction(s) with very_low confidence',
  'very_low',
  ['biochemical_interaction'],
  ['Extracellular Matrix Organization']),
 'hallmark': ('Genes co-regulate 1 cancer hallmark process(es) with low confidence',
  'low',
  ['apoptotic_regulation'],
  ['Apoptosis']),
 'function': ('Genes do not share molecular functions', 'none', [], []),
 'process': ('Genes participate in 5 shared biological process(es) with high confidence',
  'high',
  ['developmental_process'],
  ['Negative Regulation Of Low-Density Lipoprotein Receptor Activity (GO:1905598)',
   'Neu

In [24]:
from medea.tool_space.open_scholar import *
search_paper_via_query("alzheimers")

[{'paperId': 'fcf1910d71bb46759a0233e14efcad63d7d7007c',
  'externalIds': {'PubMedCentral': '11191496',
   'DOI': '10.3233/JAD-230772',
   'CorpusId': 270002290,
   'PubMed': '38788067'},
  'publicationVenue': {'id': '6c8289bb-af7a-4fe0-9174-7850c42777a9',
   'name': "Journal of Alzheimer's Disease",
   'type': 'journal',
   'alternate_names': ["J Alzheimer's Dis",
    'J Alzheimers Dis',
    'Journal of Alzheimers Disease'],
   'issn': '1387-2877',
   'url': 'http://content.iospress.com/journals/journal-of-alzheimers-disease/',
   'alternate_urls': ['https://www.j-alz.com/', 'http://www.j-alz.com/']},
  'url': 'https://www.semanticscholar.org/paper/fcf1910d71bb46759a0233e14efcad63d7d7007c',
  'title': 'A Scoping Review of Alzheimers Disease Hypotheses: An Array of Uni- and Multi-Factorial Theories',
  'year': 2024,
  'citationCount': 11,
  'openAccessPdf': {'url': 'https://content.iospress.com:443/download/journal-of-alzheimers-disease/jad230772?id=journal-of-alzheimers-disease%2Fjad2

In [27]:
from medea.tool_space.open_scholar import *
search_paper_via_query("alzheimers",max_paper_num=5,attempt=5)

[{'paperId': 'fcf1910d71bb46759a0233e14efcad63d7d7007c',
  'externalIds': {'PubMedCentral': '11191496',
   'DOI': '10.3233/JAD-230772',
   'CorpusId': 270002290,
   'PubMed': '38788067'},
  'publicationVenue': {'id': '6c8289bb-af7a-4fe0-9174-7850c42777a9',
   'name': "Journal of Alzheimer's Disease",
   'type': 'journal',
   'alternate_names': ["J Alzheimer's Dis",
    'J Alzheimers Dis',
    'Journal of Alzheimers Disease'],
   'issn': '1387-2877',
   'url': 'http://content.iospress.com/journals/journal-of-alzheimers-disease/',
   'alternate_urls': ['https://www.j-alz.com/', 'http://www.j-alz.com/']},
  'url': 'https://www.semanticscholar.org/paper/fcf1910d71bb46759a0233e14efcad63d7d7007c',
  'title': 'A Scoping Review of Alzheimers Disease Hypotheses: An Array of Uni- and Multi-Factorial Theories',
  'year': 2024,
  'citationCount': 11,
  'openAccessPdf': {'url': 'https://content.iospress.com:443/download/journal-of-alzheimers-disease/jad230772?id=journal-of-alzheimers-disease%2Fjad2